In [46]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

True

In [47]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
parser = StrOutputParser()

In [48]:
class SentimentState(TypedDict):

    text : str
    sentiment : str
    issue : str
    negative_response : str
    positive_response : str


In [49]:
# define the schema
class SentimentSchema(BaseModel):

    sentiment : Literal["positive", "negative"] = Field(description="Sentiment of the review")

In [50]:
structured_llm = llm.with_structured_output(SentimentSchema)

In [51]:
def find_sentiment(state : SentimentState) -> SentimentState:

    text = state['text']

    result = structured_llm.invoke(f"What is the sentiment of given text \n {text}")

    return {"sentiment" : result.sentiment}



def run_diagnosis(state : SentimentState) -> SentimentState:

    text = state['text']

    ans = llm.invoke(f"what is the real issue that complained by user here \n {text}")
    result = parser.invoke(ans)
    
    return {"issue" : result}

def n_res(state : SentimentState) -> SentimentState:

    issue = state['issue']

    ans = llm.invoke(f"generate the 2-3 line response for user who affected with some issues \n user review : {state['text']} and issue is {issue}")
    result = parser.invoke(ans)

    return {"negative_response" : result}


def p_res(state : SentimentState) -> SentimentState:

    ans = llm.invoke(f"generate the 2-3 line response for the user positive review in response of below text :: \n {state['text']}")
    result = parser.invoke(ans)

    return {"positive_response" : result}


In [52]:
def check_condition(state : SentimentState) -> Literal["run_diagnosis","p_res"]:

    if state["sentiment"].lower() == "negative":
        return "run_diagnosis"
    else:
        return "p_res"

In [53]:
graph = StateGraph(SentimentState)

graph.add_node("find_sentiment",find_sentiment)
graph.add_node("run_diagnosis",run_diagnosis)
graph.add_node("n_res",n_res)
graph.add_node("p_res",p_res)

graph.add_edge(START,"find_sentiment")
graph.add_conditional_edges("find_sentiment",check_condition)    # <---- main logic is here for condition.
graph.add_edge("run_diagnosis","n_res")
graph.add_edge("n_res",END)
graph.add_edge("p_res",END)

workflow = graph.compile()

In [54]:
# execute the graph

review = input("Enter the rwview here :: ")
initial_state = {"text" : review}

final_state = workflow.invoke(initial_state)

print(final_state)

{'text': 'This phone gets very hot when playing games, and the battery drains too fast. The screen also freezes a lot, making it hard to use. I am very sad with this poor quality and want my money back.', 'sentiment': 'negative', 'issue': 'The real issue complained about by the user is a combination of **significant performance and quality defects** with the phone, leading to a **poor and frustrating user experience**.\n\nSpecifically, the core problems are:\n\n1.  **Performance Issues:**\n    *   **Overheating** during use (especially gaming).\n    *   **Frequent screen freezing**, making the device unusable.\n2.  **Battery Issues:**\n    *   **Rapid battery drain**.\n3.  **Overall Quality/Reliability:**\n    *   The user perceives the phone as having **"poor quality"** because it fails to perform basic functions reliably and consistently.\n\nUltimately, the user feels the phone is **not fit for purpose** and is seeking a **refund** due to its unacceptable functionality and quality.',